In [148]:
from vae import*
from utils import*
import pandas as pd
#from data_builder_no_labels import DataBuilder
from torch.utils.data import DataLoader
import sys
import os 
from collections import Counter

In [149]:
from utils import*
from torch.utils.data import Dataset
import torch

class DataBuilder(Dataset):
    def __init__(self, data):
        self.features = data.iloc[:, 1:-1].values.astype('float32')
        self.mouse_ids = data['MouseID']

    def __getitem__(self, index):
        return self.features[index], self.mouse_ids[index]
    
    def __len__(self):
        return len(self.features)

In [150]:
ex = pd.read_csv('/Users/racheliritani/Desktop/AD Project/VAE-work/c-gmvae/nature_filtered_nonan_nocfc.csv')
#ex.iloc[:, 1:-1]
ex

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y2_Wheel_AvgDistLFC,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet
0,DO-40-2126,26,1541,29.523513,31.341861,27.520384,161.603172,84.828525,53.462304,516.996301,...,-0.103605,0.140082,197.287795,171.740018,99.647222,0.051882,0.172374,0.166289,1,40
1,DO-40-2117,26,1533,25.635280,28.483440,28.120159,146.162810,68.658928,67.881061,449.140753,...,1.872995,0.931612,194.905295,154.168518,177.974722,0.189739,0.177600,0.248073,1,40
2,DO-40-2118,26,1477,36.058736,37.270076,29.312845,154.814065,102.302694,67.881061,506.097998,...,0.162894,0.241456,181.205295,169.655518,173.209722,0.075896,0.215548,0.286021,1,40
3,DO-40-2115,26,1465,26.111027,24.494062,20.748181,152.891564,82.116435,55.384805,478.315657,...,2.604665,3.152223,160.655795,138.979518,111.858222,0.250546,0.181486,0.232985,1,40
4,DO-40-2070,24,1540,19.467849,20.099654,20.155893,136.404410,84.241603,96.770388,524.934223,...,1.949493,2.261991,129.318957,137.733391,135.754764,0.114761,0.177377,0.263152,1,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,DO-AL-0009,22,349,21.131405,63.024938,67.590983,140.940870,255.344186,261.476515,491.072949,...,-6.656427,-2.736010,250.583787,66.013018,58.352082,0.301015,0.480959,0.555556,5,AL
429,DO-1D-3015,22,309,30.660285,63.024938,67.590983,152.928679,255.344186,261.476515,681.818182,...,-6.656427,-2.736010,163.396287,66.013018,58.352082,0.301015,0.480959,0.555556,4,1D
430,DO-2D-4010,22,309,20.517814,63.024938,67.590983,144.277424,255.344186,261.476515,681.818182,...,-6.656427,-2.736010,173.202287,66.013018,58.352082,0.301015,0.480959,0.555556,3,2D
431,DO-1D-3006,22,244,26.757601,63.024938,67.590983,112.556159,255.344186,261.476515,681.818182,...,-6.656427,-2.736010,104.987287,66.013018,58.352082,0.301015,0.480959,0.555556,4,1D


In [151]:
trained_model_path = '/Users/racheliritani/Desktop/AD Project/mm-vae/results_DIET/CGMVAE_cfc_cond_5bins_5dim/saved_model_epoch5000.pth'
data_no_cfc_path = '/Users/racheliritani/Desktop/AD Project/VAE-work/c-gmvae/nature_filtered_nonan_nocfc.csv'
output_path = '/Users/racheliritani/Desktop/AD Project/mm-vae/cfc_pred/pseudo_labels_no_cfc.csv'
os.makedirs(output_path, exist_ok=True)
accelerator = True 

if accelerator:
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
all_data = pd.read_csv(data_no_cfc_path)

test_set = DataBuilder(all_data)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)
model = torch.load(trained_model_path, map_location=device)
model.eval()

Using device: mps


CGMVAE(
  (label): Embedding(5, 1)
  (fc1): Linear(in_features=37, out_features=16, bias=True)
  (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc21): Linear(in_features=16, out_features=25, bias=True)
  (fc22): Linear(in_features=16, out_features=25, bias=True)
  (fc3): Linear(in_features=6, out_features=16, bias=True)
  (bn3): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc4): Linear(in_features=16, out_features=36, bias=True)
)

In [152]:
gmm_centers = pd.read_csv('/Users/racheliritani/Desktop/AD Project/VAE-work/c-gmvae/results_DIET/5_gmm_center_5dims.csv')
gmm_centers = gmm_centers.iloc[:, 1:]
gmm_centers

,center1,center2,center3,center4,center5
0,-20.172264,-11.293541,-7.190775,-3.983031,9.463689
1,-22.702984,-20.301035,-12.269944,-6.019031,6.202443
2,-11.748712,-8.682820,-0.457847,7.194811,16.704256
3,-12.459927,-6.230437,-2.124611,6.765072,18.014086
4,2.927976,5.914355,10.095953,21.043993,33.145348


In [154]:
labels_to_centers = np.array([0, 3, 2, 4, 1])

In [155]:
pseudo_labels = []
for batch_idx, (data, mouse_ids) in enumerate(test_loader):
    data = data.to(device)
    print(f'batch (sample) {batch_idx}: ')
    print(mouse_ids)
    #if batch_idx == 20:
    distance_to_centers = []
    for i in range(5):
        #print(f'label {i}')
        center_mapping = np.where(labels_to_centers == i)
        #print(f'associated_gmm_center_idx:{center_mapping[0][0]}')
        gmm_center = gmm_centers.iloc[:, center_mapping[0][0]]
        gmm_center_torch = torch.tensor(gmm_center.values, dtype=torch.float32)
        gmm_center_torch = gmm_center_torch.to(device)
        labels = torch.full((data.shape[0],), i, dtype=torch.int64)
        labels = labels.to(device)
        mu, logvar = model.encode(data, labels)
        # print(mu)
        #print('mu:')
        #print(mu[:, i])
        #print('center:')
        #print(gmm_center_torch)
        label_center = gmm_center_torch
        point_center = mu[:, i, :]
        diff = label_center-point_center
        squared_sum = torch.sum(diff.pow(2))
        label_dist = torch.sqrt(squared_sum)
        distance_to_centers.append(label_dist.item())
        #print('distance: ')
        #print(label_dist)
    #print(distance_to_centers)
    pseudo_label = np.argmin(np.array(distance_to_centers))
    pseudo_labels.append(pseudo_label)
    print(distance_to_centers)
    print(min(distance_to_centers))
    print(pseudo_label)

batch (sample) 0: 
('DO-40-2126',)
[65.16738891601562, 7.251326560974121, 24.56241798400879, 51.706199645996094, 58.49348831176758]
7.251326560974121
1
batch (sample) 1: 
('DO-40-2117',)
[65.16738891601562, 7.251326560974121, 24.56241798400879, 51.706199645996094, 58.49348831176758]
7.251326560974121
1
batch (sample) 2: 
('DO-40-2118',)
[69.96627807617188, 3.654996871948242, 26.13920021057129, 48.7436637878418, 54.38074493408203]
3.654996871948242
1
batch (sample) 3: 
('DO-40-2115',)
[69.96627807617188, 3.654996871948242, 26.13920021057129, 48.7436637878418, 54.38074493408203]
3.654996871948242
1
batch (sample) 4: 
('DO-40-2070',)
[65.16738891601562, 7.251326560974121, 24.56241798400879, 51.706199645996094, 58.49348831176758]
7.251326560974121
1
batch (sample) 5: 
('DO-AL-0118',)
[69.96627807617188, 3.654996871948242, 26.13920021057129, 48.7436637878418, 54.38074493408203]
3.654996871948242
1
batch (sample) 6: 
('DO-40-2116',)
[65.16738891601562, 7.251326560974121, 24.56241798400879, 5

In [156]:
for batch_idx, (data, mouse_ids) in enumerate(test_loader):
    print('batch_idx: ' + str(batch_idx))
    print('data: ' + str(data))
    print('mouse_id: ' + str(mouse_ids))


batch_idx: 0
data: tensor([[ 2.6000e+01,  1.5410e+03,  2.9524e+01,  3.1342e+01,  2.7520e+01,
          1.6160e+02,  8.4829e+01,  5.3462e+01,  5.1700e+02,  4.3636e+02,
          5.1253e+02,  1.5957e+04,  1.5244e+04,  1.8299e+04,  1.8508e+01,
          1.8025e+01,  1.2654e+01,  1.3187e+02,  2.0043e+02,  2.2545e+02,
          1.6444e+00,  1.4039e+00,  1.0852e+00, -1.7420e-01, -6.6592e-02,
         -1.1120e-01,  2.0675e-01, -1.0361e-01,  1.4008e-01,  1.9729e+02,
          1.7174e+02,  9.9647e+01,  5.1882e-02,  1.7237e-01,  1.6629e-01,
          1.0000e+00]])
mouse_id: ('DO-40-2126',)
batch_idx: 1
data: tensor([[2.6000e+01, 1.5330e+03, 2.5635e+01, 2.8483e+01, 2.8120e+01, 1.4616e+02,
         6.8659e+01, 6.7881e+01, 4.4914e+02, 4.8485e+02, 4.2804e+02, 2.0088e+04,
         1.5067e+04, 1.7329e+04, 1.6249e+01, 1.6511e+01, 1.5013e+01, 2.6720e+02,
         2.4364e+02, 1.6003e+02, 1.7530e+00, 1.7631e+00, 1.5935e+00, 3.4138e-02,
         3.9506e-01, 3.4892e-01, 1.1956e+00, 1.8730e+00, 9.3161e-01, 1

In [157]:
len(pseudo_labels)

433

In [158]:
counts = Counter(pseudo_labels)
print(counts)

Counter({1: 422, 2: 11})


In [122]:
all_data

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y2_Wheel_AvgDistLFC,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet
0,DO-40-2126,26,1541,29.523513,31.341861,27.520384,161.603172,84.828525,53.462304,516.996301,...,-0.103605,0.140082,197.287795,171.740018,99.647222,0.051882,0.172374,0.166289,1,40
1,DO-40-2117,26,1533,25.635280,28.483440,28.120159,146.162810,68.658928,67.881061,449.140753,...,1.872995,0.931612,194.905295,154.168518,177.974722,0.189739,0.177600,0.248073,1,40
2,DO-40-2118,26,1477,36.058736,37.270076,29.312845,154.814065,102.302694,67.881061,506.097998,...,0.162894,0.241456,181.205295,169.655518,173.209722,0.075896,0.215548,0.286021,1,40
3,DO-40-2115,26,1465,26.111027,24.494062,20.748181,152.891564,82.116435,55.384805,478.315657,...,2.604665,3.152223,160.655795,138.979518,111.858222,0.250546,0.181486,0.232985,1,40
4,DO-40-2070,24,1540,19.467849,20.099654,20.155893,136.404410,84.241603,96.770388,524.934223,...,1.949493,2.261991,129.318957,137.733391,135.754764,0.114761,0.177377,0.263152,1,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,DO-AL-0009,22,349,21.131405,63.024938,67.590983,140.940870,255.344186,261.476515,491.072949,...,-6.656427,-2.736010,250.583787,66.013018,58.352082,0.301015,0.480959,0.555556,5,AL
429,DO-1D-3015,22,309,30.660285,63.024938,67.590983,152.928679,255.344186,261.476515,681.818182,...,-6.656427,-2.736010,163.396287,66.013018,58.352082,0.301015,0.480959,0.555556,4,1D
430,DO-2D-4010,22,309,20.517814,63.024938,67.590983,144.277424,255.344186,261.476515,681.818182,...,-6.656427,-2.736010,173.202287,66.013018,58.352082,0.301015,0.480959,0.555556,3,2D
431,DO-1D-3006,22,244,26.757601,63.024938,67.590983,112.556159,255.344186,261.476515,681.818182,...,-6.656427,-2.736010,104.987287,66.013018,58.352082,0.301015,0.480959,0.555556,4,1D


In [124]:
all_data['pseudo_label'] = pseudo_labels
all_data

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y3_Wheel_AvgDistLFC,Y1A_Grip_All,Y2A_Grip_All,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet,pseudo_label
0,DO-40-2126,26,1541,29.523513,31.341861,27.520384,161.603172,84.828525,53.462304,516.996301,...,0.140082,197.287795,171.740018,99.647222,0.051882,0.172374,0.166289,1,40,0
1,DO-40-2117,26,1533,25.635280,28.483440,28.120159,146.162810,68.658928,67.881061,449.140753,...,0.931612,194.905295,154.168518,177.974722,0.189739,0.177600,0.248073,1,40,0
2,DO-40-2118,26,1477,36.058736,37.270076,29.312845,154.814065,102.302694,67.881061,506.097998,...,0.241456,181.205295,169.655518,173.209722,0.075896,0.215548,0.286021,1,40,0
3,DO-40-2115,26,1465,26.111027,24.494062,20.748181,152.891564,82.116435,55.384805,478.315657,...,3.152223,160.655795,138.979518,111.858222,0.250546,0.181486,0.232985,1,40,0
4,DO-40-2070,24,1540,19.467849,20.099654,20.155893,136.404410,84.241603,96.770388,524.934223,...,2.261991,129.318957,137.733391,135.754764,0.114761,0.177377,0.263152,1,40,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428,DO-AL-0009,22,349,21.131405,63.024938,67.590983,140.940870,255.344186,261.476515,491.072949,...,-2.736010,250.583787,66.013018,58.352082,0.301015,0.480959,0.555556,5,AL,2
429,DO-1D-3015,22,309,30.660285,63.024938,67.590983,152.928679,255.344186,261.476515,681.818182,...,-2.736010,163.396287,66.013018,58.352082,0.301015,0.480959,0.555556,4,1D,0
430,DO-2D-4010,22,309,20.517814,63.024938,67.590983,144.277424,255.344186,261.476515,681.818182,...,-2.736010,173.202287,66.013018,58.352082,0.301015,0.480959,0.555556,3,2D,3
431,DO-1D-3006,22,244,26.757601,63.024938,67.590983,112.556159,255.344186,261.476515,681.818182,...,-2.736010,104.987287,66.013018,58.352082,0.301015,0.480959,0.555556,4,1D,3


In [159]:
# save csv with pseudo labels 
# all_data.to_csv('/Users/racheliritani/Desktop/AD Project/mm-vae/cfc_pred/data_no_cfc_pseudo_labels.csv', index=False)

In [160]:
pd.read_csv('/Users/racheliritani/Desktop/AD Project/VAE-work/c-gmvae/data_with_cfc_5bins.csv')

,MouseID,Generation,SurvDays,Y1A_BW_BW,Y2A_BW_BW,Y3A_BW_BW,Y1_Glu.F_Glucose,Y2_Glu.F_Glucose,Y3_Glu.F_Glucose,Y1_Echo_BPM,...,Y3A_Grip_All,Y1A_Frailty_FrailtyAdj,Y2A_Frailty_FrailtyAdj,Y3A_Frailty_FrailtyAdj,Diet_num,Diet_x,Diet_y,Mouse.ID,CFC.24mo.Average,cfc_bin
0,DO-40-2162,28,1611,24.787142,26.993230,23.645309,164.713839,134.572768,74.479328,499.174601,...,179.558615,0.032865,0.133616,0.159587,1,40,40,DO-40-2162,3.78,0
1,DO-40-2179,28,1557,30.432753,23.594599,23.529931,161.043852,71.901384,69.226421,495.841743,...,208.833723,0.183285,0.231782,0.235033,1,40,40,DO-40-2179,26.71,1
2,DO-40-2181,28,1533,20.747600,19.072890,20.528352,161.043852,93.048894,60.575167,520.949018,...,158.799723,0.069442,0.231782,0.197085,1,40,40,DO-40-2181,27.02,1
3,DO-40-2187,28,1520,25.977819,29.254446,30.242503,127.747193,85.358891,71.148922,466.210754,...,208.833223,0.106295,0.174654,0.216059,1,40,40,DO-40-2187,50.40,2
4,DO-40-2099,26,1638,24.242912,23.626717,22.912834,166.520150,72.692867,59.960123,464.789558,...,153.357082,0.110173,0.155822,0.285573,1,40,40,DO-40-2099,34.18,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,DO-2D-4007,22,771,40.141463,46.980545,67.590983,112.556159,165.312618,261.476515,554.545551,...,58.352082,0.301015,0.209083,0.555556,3,2D,2D,DO-2D-4007,23.69,1
500,DO-1D-3030,22,748,24.315156,29.916707,67.590983,139.326564,136.439061,261.476515,544.317015,...,58.352082,0.301015,0.082493,0.555556,4,1D,1D,DO-1D-3030,29.78,1
501,DO-20-1021,22,738,30.809768,38.120260,67.590983,168.911742,152.502965,261.476515,542.558402,...,58.352082,0.301015,0.224366,0.555556,2,20,20,DO-20-1021,18.22,0
502,DO-AL-0002,22,755,30.261862,48.841393,67.590983,139.018370,125.113095,261.476515,477.144237,...,58.352082,0.301015,0.277963,0.555556,5,AL,AL,DO-AL-0002,57.42,2
